Прогнозирование исхода процедуры на основе двух изображений до и после.

In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import convnext_small, ConvNeXt_Small_Weights

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score


In [2]:
CSV_PATH = Path(r"D:\diplom\dataset_before_after\cropped_pairs\before_after_pairs_cropped.csv")
WEIGHTS_DIR = Path(r"D:\diplom\weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

BEST_WEIGHTS_PATH = WEIGHTS_DIR / "siamese_convnext_small_before_after_v1_cropped.pth"

IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-4
VAL_SIZE = 0.2
RANDOM_STATE = 42
NUM_WORKERS = 0   

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(RANDOM_STATE)

In [4]:
def normalize_binary_value(x):
    if pd.isna(x):
        return None

    s = str(x).strip().lower().replace(",", ".")
    mapping = {
        "0": 0,
        "0.0": 0,
        "false": 0,
        "no": 0,
        "нет": 0,
        "1": 1,
        "1.0": 1,
        "true": 1,
        "yes": 1,
        "да": 1,
    }
    return mapping.get(s, None)


def load_dataframe(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        raise FileNotFoundError(f"Не найден CSV: {csv_path}")

    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    required_cols = ["video_file", "before_path", "after_path", "result"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"В CSV отсутствуют столбцы: {missing}")

    df = df.copy()
    df["video_file"] = df["video_file"].astype(str).str.strip()
    df["before_path"] = df["before_path"].astype(str).str.strip()
    df["after_path"] = df["after_path"].astype(str).str.strip()

    df = df[(df["before_path"] != "") & (df["after_path"] != "")]
    df = df[df["before_path"].apply(lambda p: Path(p).exists())]
    df = df[df["after_path"].apply(lambda p: Path(p).exists())]

    df["label"] = df["result"].apply(normalize_binary_value)
    df = df[df["label"].notna()].copy()
    df["label"] = df["label"].astype(int)

    unique_labels = sorted(df["label"].unique().tolist())
    if unique_labels != [0, 1]:
        raise ValueError(f"Ожидались бинарные метки 0/1. Найдено: {unique_labels}")

    if len(df) == 0:
        raise ValueError("После фильтрации не осталось валидных строк.")

    return df.reset_index(drop=True)


df = load_dataframe(CSV_PATH)

print(f"Всего валидных пар: {len(df)}")
print("Распределение классов:")
print(df["label"].value_counts().sort_index())

Всего валидных пар: 339
Распределение классов:
label
0     67
1    272
Name: count, dtype: int64


In [5]:
groups = df["video_file"].values
y = df["label"].values

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=VAL_SIZE,
    random_state=RANDOM_STATE
)

train_idx, val_idx = next(gss.split(df, y=y, groups=groups))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"\nTrain samples: {len(train_df)}")
print(f"Val samples:   {len(val_df)}")
print(f"Train videos:  {train_df['video_file'].nunique()}")
print(f"Val videos:    {val_df['video_file'].nunique()}")


Train samples: 271
Val samples:   68
Train videos:  271
Val videos:    68


In [6]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

In [7]:
class BeforeAfterDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        before_img = Image.open(row["before_path"]).convert("RGB")
        after_img = Image.open(row["after_path"]).convert("RGB")

        if self.transform is not None:
            before_img = self.transform(before_img)
            after_img = self.transform(after_img)

        label = torch.tensor(row["label"], dtype=torch.float32)
        return before_img, after_img, label


train_ds = BeforeAfterDataset(train_df, transform=transform)
val_ds = BeforeAfterDataset(val_df, transform=transform)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [8]:
class SiameseConvNeXtSmall(nn.Module):
    def __init__(self, dropout=0.35):
        super().__init__()

        backbone = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)

        feat_dim = backbone.classifier[2].in_features
        backbone.classifier[2] = nn.Identity()
        self.backbone = backbone

        # Более богатое объединение признаков:
        # before, after, |after-before|, before*after
        fusion_dim = feat_dim * 4

        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(1024, 256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 1),
        )

    def forward_once(self, x):
        return self.backbone(x)  # [B, feat_dim]

    def forward(self, before, after):
        f_before = self.forward_once(before)
        f_after = self.forward_once(after)

        f_absdiff = torch.abs(f_after - f_before)
        f_prod = f_before * f_after

        fused = torch.cat([f_before, f_after, f_absdiff, f_prod], dim=1)
        logits = self.head(fused).squeeze(1)
        return logits


model = SiameseConvNeXtSmall().to(DEVICE)

In [9]:
num_pos = int((train_df["label"] == 1).sum())
num_neg = int((train_df["label"] == 0).sum())

if num_pos == 0 or num_neg == 0:
    raise ValueError("В train должен быть хотя бы один объект каждого класса.")

pos_weight_value = num_neg / max(num_pos, 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)


In [10]:
def compute_metrics(y_true, logits):
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(np.int64)

    acc = accuracy_score(y_true, preds)
    f1 = f1_score(y_true, preds, zero_division=0)

    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, probs)
    else:
        auc = float("nan")

    return acc, f1, auc


def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    all_labels = []
    all_logits = []

    for before, after, labels in loader:
        before = before.to(DEVICE, non_blocking=True)
        after = after.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(is_train):
            logits = model(before, after)
            loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.detach().cpu().numpy().tolist())
        all_logits.extend(logits.detach().cpu().numpy().tolist())

    epoch_loss = total_loss / len(loader.dataset)
    acc, f1, auc = compute_metrics(np.array(all_labels), np.array(all_logits))
    return epoch_loss, acc, f1, auc

In [11]:
best_score = -1.0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1, train_auc = run_epoch(
        train_loader, model, criterion, optimizer=optimizer
    )

    val_loss, val_acc, val_f1, val_auc = run_epoch(
        val_loader, model, criterion, optimizer=None
    )

    print(
        f"[{epoch:02d}/{EPOCHS}] "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.4f} "
        f"train_f1={train_f1:.4f} "
        f"train_auc={train_auc:.4f} | "
        f"val_loss={val_loss:.4f} "
        f"val_acc={val_acc:.4f} "
        f"val_f1={val_f1:.4f} "
        f"val_auc={val_auc:.4f}"
    )

    score = val_auc if not np.isnan(val_auc) else val_f1
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), BEST_WEIGHTS_PATH)

[01/15] train_loss=0.2579 train_acc=0.3838 train_f1=0.4732 train_auc=0.4543 | val_loss=0.2977 val_acc=0.7500 val_f1=0.8571 val_auc=0.4256
[02/15] train_loss=0.2577 train_acc=0.5498 train_f1=0.6720 train_auc=0.5348 | val_loss=0.2904 val_acc=0.2500 val_f1=0.0377 val_auc=0.4487
[03/15] train_loss=0.2595 train_acc=0.4244 train_f1=0.5301 train_auc=0.4369 | val_loss=0.2948 val_acc=0.7500 val_f1=0.8571 val_auc=0.3852
[04/15] train_loss=0.2588 train_acc=0.5535 train_f1=0.6905 train_auc=0.4483 | val_loss=0.3041 val_acc=0.7500 val_f1=0.8571 val_auc=0.4913
[05/15] train_loss=0.2587 train_acc=0.4649 train_f1=0.5773 train_auc=0.4684 | val_loss=0.2988 val_acc=0.7500 val_f1=0.8571 val_auc=0.6067
[06/15] train_loss=0.2603 train_acc=0.4982 train_f1=0.6114 train_auc=0.5036 | val_loss=0.2863 val_acc=0.2500 val_f1=0.0000 val_auc=0.5571
[07/15] train_loss=0.2585 train_acc=0.4244 train_f1=0.5125 train_auc=0.5007 | val_loss=0.2991 val_acc=0.7500 val_f1=0.8571 val_auc=0.5825
[08/15] train_loss=0.2569 train_ac

In [12]:
model.load_state_dict(torch.load(BEST_WEIGHTS_PATH, map_location=DEVICE))

final_val_loss, final_val_acc, final_val_f1, final_val_auc = run_epoch(
    val_loader, model, criterion, optimizer=None
)

print("\nЛучшие сохранённые веса:")
print(BEST_WEIGHTS_PATH)

print("\nИтоговые метрики на validation:")
print(f"val_loss = {final_val_loss:.4f}")
print(f"val_acc  = {final_val_acc:.4f}")
print(f"val_f1   = {final_val_f1:.4f}")
print(f"val_auc  = {final_val_auc:.4f}")


Лучшие сохранённые веса:
D:\diplom\weights\siamese_convnext_small_before_after_v1_cropped.pth

Итоговые метрики на validation:
val_loss = 0.2930
val_acc  = 0.7500
val_f1   = 0.8571
val_auc  = 0.6828
